In [1]:
!pip install transformers==4.46.0 accelerate==1.1.1 -U bitsandbytes

In [2]:
import transformers, bitsandbytes, triton
print(transformers.__version__, bitsandbytes.__version__, triton.__version__)

4.46.0 0.50.2 3.6.0


In [3]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
# expect: Tesla T4, 15360 MiB

Tesla T4, 15360 MiB


In [4]:
import threading, time
import pynvml

pynvml.nvmlInit()
_gpu_handle = pynvml.nvmlDeviceGetHandleByIndex(0)

_sampler_thread = None
_sampler_stop_event = threading.Event()
_util_samples = []


def _poll_util():
    while not _sampler_stop_event.is_set():
        util = pynvml.nvmlDeviceGetUtilizationRates(_gpu_handle).gpu
        _util_samples.append(util)
        time.sleep(0.05)


def start_sampler():
    global _sampler_thread
    _util_samples.clear()
    _sampler_stop_event.clear()
    _sampler_thread = threading.Thread(target=_poll_util, daemon=True)
    _sampler_thread.start()


def stop_sampler():
    _sampler_stop_event.set()
    if _sampler_thread is not None:
        _sampler_thread.join()


def read_util_mean():
    if not _util_samples:
        return 0.0
    return sum(_util_samples) / len(_util_samples)

In [5]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

In [6]:
import time, gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import triton

tok = AutoTokenizer.from_pretrained(MODEL)

def load(dtype: str):
    if dtype == "fp16":
        return AutoModelForCausalLM.from_pretrained(
            MODEL, torch_dtype=torch.float16, device_map="cuda")
    if dtype == "int8":
        qc = BitsAndBytesConfig(load_in_8bit=True)
        return AutoModelForCausalLM.from_pretrained(
            MODEL, quantization_config=qc, device_map="cuda")
    raise ValueError(dtype)

def make_prompt(context_tokens: int) -> str:
    # a filler prompt padded to about context_tokens input tokens
    base = "Summarise the following text in one sentence.\n"
    filler = ("The data center runs many small inference requests all day. " * 400)
    ids = tok(base + filler)["input_ids"][:context_tokens]
    return tok.decode(ids)

def resident_vram_gb() -> float:
    torch.cuda.synchronize()
    return torch.cuda.memory_reserved() / (1024 ** 3)

def profile(model, dtype: str, context: int, new_tokens: int = 128, batch: int = 1):
    prompt = make_prompt(context)
    prompts = [prompt] * batch
    enc = tok(prompts, return_tensors="pt", padding=True).to("cuda")
    # warm-up (compile/allocate), not measured
    _ = model.generate(**enc, max_new_tokens=8, do_sample=False)
    vram = resident_vram_gb()
    start_sampler()
    t0 = time.time()
    out = model.generate(**enc, max_new_tokens=new_tokens, do_sample=False)
    dt = time.time() - t0
    stop_sampler()
    gen_tokens = (out.shape[1] - enc["input_ids"].shape[1]) * batch
    return {
        "dtype": dtype,
        "context": context,
        "vram_gb": round(vram, 3),
        "util_mean": round(read_util_mean(), 1),
        "tokens_per_s": round(gen_tokens / dt, 1),
    }

def free_vram():
    """Hand freed memory back to the driver.

    Delete the model variable yourself first, in the cell, with `del model`.
    Passing it to a helper does not work: the helper deletes its own local name
    while your notebook variable still holds the weights, so nothing is freed
    and empty_cache() has nothing to give back.
    """
    gc.collect()
    torch.cuda.empty_cache()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
rows = []
for dtype in ["fp16", "int8"]:
    model = load(dtype)
    for context in [512, 2048, 4096]:
        row = profile(model, dtype, context)
        print(row)
        rows.append(row)
    del model      # without this the next dtype loads on top of this one
    free_vram()

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


{'dtype': 'fp16', 'context': 512, 'vram_gb': 3.113, 'util_mean': 56.1, 'tokens_per_s': 30.7}
{'dtype': 'fp16', 'context': 2048, 'vram_gb': 3.295, 'util_mean': 67.6, 'tokens_per_s': 27.8}
{'dtype': 'fp16', 'context': 4096, 'vram_gb': 3.568, 'util_mean': 82.3, 'tokens_per_s': 24.8}
{'dtype': 'int8', 'context': 512, 'vram_gb': 1.805, 'util_mean': 26.6, 'tokens_per_s': 6.0}
{'dtype': 'int8', 'context': 2048, 'vram_gb': 2.035, 'util_mean': 30.4, 'tokens_per_s': 5.9}
{'dtype': 'int8', 'context': 4096, 'vram_gb': 2.309, 'util_mean': 35.2, 'tokens_per_s': 5.4}


In [12]:
model = load("fp16")
b1 = profile(model, "fp16", 512, new_tokens=128, batch=1)
b8 = profile(model, "fp16", 512, new_tokens=128, batch=8)
del model
free_vram()
print("batch 1:", b1)
print("batch 8:", b8)
print("tokens/s ratio:", round(b8["tokens_per_s"] / b1["tokens_per_s"], 2))
print("util delta:", round(b8["util_mean"] - b1["util_mean"], 1))

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


batch 1: {'dtype': 'fp16', 'context': 512, 'vram_gb': 3.113, 'util_mean': 51.9, 'tokens_per_s': 27.9}
batch 8: {'dtype': 'fp16', 'context': 512, 'vram_gb': 3.527, 'util_mean': 91.4, 'tokens_per_s': 200.5}
tokens/s ratio: 7.19
util delta: 39.5


In [13]:
model = load("int8")
b1_int = profile(model, "int8", 512, new_tokens=128, batch=1)
b8_int = profile(model, "int8", 512, new_tokens=128, batch=8)
del model
free_vram()
print("batch 1:", b1)
print("batch 8:", b8)
print("tokens/s ratio:", round(b8["tokens_per_s"] / b1["tokens_per_s"], 2))
print("util delta:", round(b8["util_mean"] - b1["util_mean"], 1))

batch 1: {'dtype': 'fp16', 'context': 512, 'vram_gb': 3.113, 'util_mean': 51.9, 'tokens_per_s': 27.9}
batch 8: {'dtype': 'fp16', 'context': 512, 'vram_gb': 3.527, 'util_mean': 91.4, 'tokens_per_s': 200.5}
tokens/s ratio: 7.19
util delta: 39.5


In [14]:
import json
with open("batch_check.json", "w") as f:
    json.dump({"batch1_tokens_per_s": b1["tokens_per_s"],
               "batch8_tokens_per_s": b8["tokens_per_s"]}, f, indent=4)

In [15]:
import json
with open("profile.json", "w") as f:
    json.dump(rows, f, indent=2)
print("wrote", len(rows), "rows to profile.json")

wrote 6 rows to profile.json
